### Connecting to Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive/')

Mounted at /content/drive/


### Installing the dependencies

In [ ]:
%%capture
# ICL does not use the vector index, so we only need the OpenAI client.
!pip install openai
!pip install tiktoken

### Setting up the API

In [ ]:
import os
import json
import re
import pandas as pd
import openai

# The key tells OpenAI who is making the request.
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

print("API key loaded:", os.environ["OPENAI_API_KEY"][:7] + "...")

API key loaded: sk-proj...


### Load the dataset

In [ ]:
# The dataset came with the repository cloned in notebook 1.
dataset_path = "/content/drive/MyDrive/TraCR_RAG_Fresh/author_repository/data/dataset.jsonl"

# One JSON object per line, so read it line by line.
with open(dataset_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)

print("Rows:", len(df))
print("Columns:", list(df.columns))
print("First flow:", df["flow_id"][0])
print("Its labels:", df["str_label"][0])

Rows: 433
Columns: ['flow_id', 'str_label', 'TEXT-flow_fn_threat']
First flow: 2B_TM16-signal_system_configuration
Its labels: ['T1040', 'T1059', 'T1078', 'T1105', 'T1190', 'T1195', 'T1495', 'T1552', 'T1557', 'T1565']


### Useful functions

In [ ]:
import tiktoken


def count_tokens(text):
    """Count how many tokens a piece of text will cost."""
    encoder = tiktoken.encoding_for_model("gpt-4o-mini")
    return len(encoder.encode(text))


def parse_mitre_techniques(response):
    """Pull every technique ID (like T1234) out of the model's answer."""
    mitre_pattern = r'T\d{4}'
    return re.findall(mitre_pattern, response)


def calculate_metrics(true_labels, predicted_labels):
    """Compare one prediction against the correct answer."""
    true_set = set(true_labels)
    predicted_set = set(predicted_labels)

    tp = len(true_set & predicted_set)     # correct ones it found
    fp = len(predicted_set - true_set)     # ones it invented
    fn = len(true_set - predicted_set)     # ones it missed

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}


# Quick check that all three work.
print("Tokens in one flow description:", count_tokens(df["TEXT-flow_fn_threat"][0]))
print("Parsed IDs:", parse_mitre_techniques("I think T1040 and T1557 apply here."))
print("Metrics:", calculate_metrics(['T1040', 'T1557', 'T1190'], ['T1040', 'T1557']))

Tokens in one flow description: 1287
Parsed IDs: ['T1040', 'T1557']
Metrics: {'tp': 2, 'fp': 0, 'fn': 1, 'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8}


### Zero shot prompting

In [ ]:
client = openai.OpenAI()

# The 63 techniques the model is allowed to choose from.
TECHNIQUE_LIST = """['T1495','T1485','T1595','T1134','T1040','T1132','T1098','T1069','T1036','T1562','T1187','T1486','T1119','T1027','T1498','T1654','T1548','T1082','T1552','T1614','T1531','T1204','T1529','T1046','T1489','T1195','T1566','T1659','T1059','T1213','T1133','T1080','T1005','T1078','T1001','T1190','T1203','T1136','T1491','T1033','T1189','T1068','T1652','T1049','T1020','T1041','T1021','T1105','T1518','T1200','T1053','T1557','T1056','T1087','T1565','T1499','T1657','T1559','T1074','T1106','T1560', 'T1556', 'T1589']"""


def query_gpt(prompt):
    """Send one prompt to the model and return its answer as text."""
    try:
        gpt_response = client.chat.completions.create(
            model="gpt-4o-mini-2024-07-18",
            messages=[
                {"role": "system",
                 "content": '''You are a helpful assisstant. Your name is Transportation Security AI. Your role is to help with transportation security.
                Their are some instructions in each prompt. Follow those instructions strictly.'''},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,     # no randomness, so the same input gives the same answer
            max_tokens=6000,
            top_p=0,
        )
        return str(gpt_response.choices[0].message.content)
    except Exception as e:
        print("Error querying GPT:", e)
        return None


def build_zero_shot_prompt(flow_description):
    """Their exact zero shot wording, kept word for word."""
    return f'''I am trying to do a multilabel classification of information flow description from Intelligent Transportation System (ITS) to MITRE ATT&CK Techniques.
                Here we have information flow name, its initiator and acceptor (Physcial Object), functional objects and process description associated with it, security characteristics of the information flow and the description of the information flow itself.
                We also have the STRIDE based threat information for the information flow.
                An attacker may attempt to compromise the integrity, confidentiality, or availability of the information flow in many ways.
                Find all the relevant MITRE ATT&CK techniques that the attacker might use to attack the information flow.

                Follow the instructions below carefully.

                1. We have a predefined list of MITRE ATT&CK Techniques that consists of 63 MITRE Techniques. You have to choose only the relevant MITRE ATT&CK Techniques from this list that is relevant to the information flow given.

                2. Understand the entire context, then generate a sublist of MITRE ATT&CK Technique from the given list.

                3. Do not add any other description in your answer.

                4. Only return the Technique IDs in python list format

                Given MITRE Technique List = {TECHNIQUE_LIST}
                You should return MITRE Techniques from this list.

                Here is the information flow description:
                {flow_description}

                Which are the relevant MITRE ATT&CK Techniques from the given list that the attacker might use to attack the information flow? Return the Technique IDs in python list format.
                '''


# How many flows to run. Start small. The full dataset is 433.
HOW_MANY = 20

zero_shot_predictions = []

for i in range(HOW_MANY):
    flow_description = df["TEXT-flow_fn_threat"][i]
    prompt = build_zero_shot_prompt(flow_description)

    answer = query_gpt(prompt)
    predicted = parse_mitre_techniques(answer)
    zero_shot_predictions.append(predicted)

    print(i + 1, df["flow_id"][i], "->", len(predicted), "techniques")

print()
print("Done.", len(zero_shot_predictions), "flows classified.")

1 2B_TM16-signal_system_configuration -> 13 techniques
2 2A_TM16-traffic_detector_coordination -> 9 techniques
3 2A_TM16-traffic_detector_coordination -> 9 techniques
4 2B_TM16-reversible_lane_control -> 8 techniques
5 2B_TM16-video_surveillance_control -> 9 techniques
6 2B_TM16-signal_control_device_configuration -> 8 techniques
7 _TM16-vehicle_characteristics -> 11 techniques
8 2B_TM16-traffic_detector_control -> 7 techniques
9 2A_TM16-lane_management_coordination -> 8 techniques
10 2A_TM16-lane_management_coordination -> 8 techniques
11 2B_TM16-signal_control_plans -> 10 techniques
12 _TM16-traffic_operator_data -> 9 techniques
13 2A_TM16-signal_control_coordination -> 8 techniques
14 2A_TM16-signal_control_coordination -> 11 techniques
15 2B_TM16-traffic_image_meta_data -> 10 techniques
16 2B_TM16-reversible_lane_status -> 8 techniques
17 _PT04-traveler_interface_updates -> 12 techniques
18 2C_PT09-traffic_control_priority_status -> 9 techniques
19 2C_PT09-traffic_control_priority_

In [ ]:
# Look at what the model actually wrote, before parsing.
i = 1      # the second flow, one that looks like it echoed the list

prompt = build_zero_shot_prompt(df["TEXT-flow_fn_threat"][i])
raw_answer = query_gpt(prompt)

print("RAW ANSWER:")
print(raw_answer)
print()
print("PARSED:", parse_mitre_techniques(raw_answer))
print("TRUE:  ", df["str_label"][i])

RAW ANSWER:
```python
['T1495', 'T1485', 'T1134', 'T1486', 'T1491', 'T1068', 'T1190', 'T1203', 'T1559']
```

PARSED: ['T1495', 'T1485', 'T1134', 'T1486', 'T1491', 'T1068', 'T1190', 'T1203', 'T1559']
TRUE:   ['T1020', 'T1040', 'T1059', 'T1105', 'T1190', 'T1495', 'T1557', 'T1565']


### What this costs

In [ ]:
INPUT_PRICE = 0.15      # dollars per 1,000,000 input tokens for gpt-4o-mini

description_tokens = [count_tokens(text) for text in df["TEXT-flow_fn_threat"]]
wrapper_tokens = count_tokens(build_zero_shot_prompt(""))

total_input = sum(description_tokens) + wrapper_tokens * len(description_tokens)

print("Flows:", len(description_tokens))
print("Average tokens per description:", int(sum(description_tokens) / len(description_tokens)))
print("Smallest:", min(description_tokens), " Largest:", max(description_tokens))
print("Instructions added to every prompt:", wrapper_tokens, "tokens")
print()
print("Cost for all %d flows:  $%.2f" % (len(description_tokens), total_input / 1_000_000 * INPUT_PRICE))

small = sum(description_tokens[:HOW_MANY]) + wrapper_tokens * HOW_MANY
print("Cost for the first %d flows: $%.4f" % (HOW_MANY, small / 1_000_000 * INPUT_PRICE))

Flows: 433
Average tokens per description: 1360
Smallest: 483  Largest: 4151
Instructions added to every prompt: 559 tokens

Cost for all 433 flows:  $0.12
Cost for the first 20 flows: $0.0054


### Score the zero shot run

In [ ]:
results = []

for i in range(HOW_MANY):
    metrics = calculate_metrics(df["str_label"][i], zero_shot_predictions[i])
    results.append(metrics)

# How many it guessed, versus how many were actually right.
avg_predicted = sum(len(p) for p in zero_shot_predictions) / HOW_MANY
avg_true = sum(len(df["str_label"][i]) for i in range(HOW_MANY)) / HOW_MANY
print("Average predicted per flow: %.1f" % avg_predicted)
print("Average true per flow:      %.1f" % avg_true)
print()

# Add every flow together, then score the run as a whole.
total_tp = sum(r["tp"] for r in results)
total_fp = sum(r["fp"] for r in results)
total_fn = sum(r["fn"] for r in results)

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Correct found (TP):", total_tp)
print("Invented (FP):    ", total_fp)
print("Missed (FN):      ", total_fn)
print()
print("Precision: %.3f" % precision)
print("Recall:    %.3f" % recall)
print("F1:        %.3f" % f1)

Average predicted per flow: 9.4
Average true per flow:      7.8

Correct found (TP): 37
Invented (FP):     152
Missed (FN):       119

Precision: 0.196
Recall:    0.237
F1:        0.214


### One shot prompting

In [ ]:
# Use the first flow as the worked example.
EXAMPLE_INDEX = 0
example_description = df["TEXT-flow_fn_threat"][EXAMPLE_INDEX]
example_labels = df["str_label"][EXAMPLE_INDEX]


def build_one_shot_prompt(flow_description):
    """Their zero shot wording, plus one worked example."""
    return f'''I am trying to do a multilabel classification of information flow description from Intelligent Transportation System (ITS) to MITRE ATT&CK Techniques.
                Here we have information flow name, its initiator and acceptor (Physcial Object), functional objects and process description associated with it, security characteristics of the information flow and the description of the information flow itself.
                We also have the STRIDE based threat information for the information flow.
                An attacker may attempt to compromise the integrity, confidentiality, or availability of the information flow in many ways.
                Find all the relevant MITRE ATT&CK techniques that the attacker might use to attack the information flow.

                Follow the instructions below carefully.

                1. We have a predefined list of MITRE ATT&CK Techniques that consists of 63 MITRE Techniques. You have to choose only the relevant MITRE ATT&CK Techniques from this list that is relevant to the information flow given.

                2. Understand the entire context, then generate a sublist of MITRE ATT&CK Technique from the given list.

                3. Do not add any other description in your answer.

                4. Only return the Technique IDs in python list format

                Given MITRE Technique List = {TECHNIQUE_LIST}
                You should return MITRE Techniques from this list.

                Here is an example:

                {example_description}

                Classified Labels in python list format:
                {example_labels}

                Here is your information flow description:
                {flow_description}

                Which are the relevant MITRE ATT&CK Techniques from the given list that the attacker might use to attack the information flow? Return the Technique IDs in python list format.
                '''


# Test on the flows after the example, never on the example itself.
test_indexes = [i for i in range(1, min(HOW_MANY + 1, len(df)))]

one_shot_predictions = []

for i in test_indexes:
    prompt = build_one_shot_prompt(df["TEXT-flow_fn_threat"][i])

    answer = query_gpt(prompt)
    predicted = parse_mitre_techniques(answer)
    one_shot_predictions.append(predicted)

    print(i, df["flow_id"][i], "->", len(predicted), "techniques")

print()
print("Done.", len(one_shot_predictions), "flows classified.")

1 2A_TM16-traffic_detector_coordination -> 10 techniques
2 2A_TM16-traffic_detector_coordination -> 10 techniques
3 2B_TM16-reversible_lane_control -> 10 techniques
4 2B_TM16-video_surveillance_control -> 10 techniques
5 2B_TM16-signal_control_device_configuration -> 10 techniques
6 _TM16-vehicle_characteristics -> 8 techniques
7 2B_TM16-traffic_detector_control -> 10 techniques
8 2A_TM16-lane_management_coordination -> 10 techniques
9 2A_TM16-lane_management_coordination -> 10 techniques
10 2B_TM16-signal_control_plans -> 10 techniques
11 _TM16-traffic_operator_data -> 7 techniques
12 2A_TM16-signal_control_coordination -> 10 techniques
13 2A_TM16-signal_control_coordination -> 10 techniques
14 2B_TM16-traffic_image_meta_data -> 10 techniques
15 2B_TM16-reversible_lane_status -> 10 techniques
16 _PT04-traveler_interface_updates -> 6 techniques
17 2C_PT09-traffic_control_priority_status -> 10 techniques
18 2C_PT09-traffic_control_priority_request -> 10 techniques
19 2B_PT09-signal_cont

### Score the one shot run

In [ ]:
one_shot_results = []

for position, i in enumerate(test_indexes):
    metrics = calculate_metrics(df["str_label"][i], one_shot_predictions[position])
    one_shot_results.append(metrics)

avg_predicted = sum(len(p) for p in one_shot_predictions) / len(test_indexes)
avg_true = sum(len(df["str_label"][i]) for i in test_indexes) / len(test_indexes)
print("Average predicted per flow: %.1f" % avg_predicted)
print("Average true per flow:      %.1f" % avg_true)
print()

total_tp = sum(r["tp"] for r in one_shot_results)
total_fp = sum(r["fp"] for r in one_shot_results)
total_fn = sum(r["fn"] for r in one_shot_results)

one_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
one_recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
one_f1 = 2 * one_precision * one_recall / (one_precision + one_recall) if (one_precision + one_recall) > 0 else 0

print("Correct found (TP):", total_tp)
print("Invented (FP):    ", total_fp)
print("Missed (FN):      ", total_fn)
print()
print("Precision: %.3f   (zero shot: %.3f)" % (one_precision, precision))
print("Recall:    %.3f   (zero shot: %.3f)" % (one_recall, recall))
print("F1:        %.3f   (zero shot: %.3f)" % (one_f1, f1))

Average predicted per flow: 9.6
Average true per flow:      7.7

Correct found (TP): 120
Invented (FP):     71
Missed (FN):       33

Precision: 0.628   (zero shot: 0.196)
Recall:    0.784   (zero shot: 0.237)
F1:        0.698   (zero shot: 0.214)


### One shot, their exact wording

In [ ]:
# Read their notebook from the clone, so the wording is exactly theirs.
their_notebook = "/content/drive/MyDrive/TraCR_RAG_Fresh/author_repository/in_context_learning/ICL.ipynb"

with open(their_notebook, "r", encoding="utf-8") as f:
    their_nb = json.load(f)

# Find their one shot cell: it shows a single example, not several.
one_shot_cell = None
for cell in their_nb["cells"]:
    if cell["cell_type"] != "code":
        continue
    source = "".join(cell["source"])
    if "Here is an example:" in source and "Here are a few examples" not in source:
        one_shot_cell = source
        break

# Pull the prompt text out from between the triple quotes.
start = one_shot_cell.index("final_prompt =")
start = one_shot_cell.index("'''", start) + 3
end = one_shot_cell.index("'''", start)
their_one_shot_template = one_shot_cell[start:end]


def build_their_one_shot_prompt(flow_description):
    """Their one shot prompt, verbatim from their own notebook."""
    return their_one_shot_template.replace("{prompt}", flow_description)


print("Characters:", len(their_one_shot_template))
print("Tokens:", count_tokens(their_one_shot_template))
print()
print(their_one_shot_template[:300])

Characters: 13817
Tokens: 2755

I am trying to do a multilabel classification of information flow description from Intelligent Transportation System (ITS) to MITRE ATT&CK Techniques.
                Here we have information flow name, its source and destination, some functional object description associated with it and the descrip


In [ ]:
their_one_shot_predictions = []

for i in test_indexes:
    prompt = build_their_one_shot_prompt(df["TEXT-flow_fn_threat"][i])

    answer = query_gpt(prompt)
    predicted = parse_mitre_techniques(answer)
    their_one_shot_predictions.append(predicted)

    print(i, df["flow_id"][i], "->", len(predicted), "techniques")

print()
print("Done.", len(their_one_shot_predictions), "flows classified.")

1 2A_TM16-traffic_detector_coordination -> 10 techniques
2 2A_TM16-traffic_detector_coordination -> 10 techniques
3 2B_TM16-reversible_lane_control -> 10 techniques
4 2B_TM16-video_surveillance_control -> 10 techniques
5 2B_TM16-signal_control_device_configuration -> 10 techniques
6 _TM16-vehicle_characteristics -> 10 techniques
7 2B_TM16-traffic_detector_control -> 10 techniques
8 2A_TM16-lane_management_coordination -> 10 techniques
9 2A_TM16-lane_management_coordination -> 10 techniques
10 2B_TM16-signal_control_plans -> 10 techniques
11 _TM16-traffic_operator_data -> 4 techniques
12 2A_TM16-signal_control_coordination -> 10 techniques
13 2A_TM16-signal_control_coordination -> 10 techniques
14 2B_TM16-traffic_image_meta_data -> 10 techniques
15 2B_TM16-reversible_lane_status -> 10 techniques
16 _PT04-traveler_interface_updates -> 4 techniques
17 2C_PT09-traffic_control_priority_status -> 10 techniques
18 2C_PT09-traffic_control_priority_request -> 10 techniques
19 2B_PT09-signal_con

### Compare the three runs

In [ ]:
# Map each row number to the prediction that run made for it.
zero_by_row   = {i: zero_shot_predictions[i] for i in range(HOW_MANY)}
theirs_by_row = {i: their_one_shot_predictions[pos] for pos, i in enumerate(test_indexes)}
ours_by_row   = {i: one_shot_predictions[pos] for pos, i in enumerate(test_indexes)}


def score_rows(by_row, rows):
    """Score one run over a given set of rows."""
    tp = fp = fn = 0
    for i in rows:
        m = calculate_metrics(df["str_label"][i], by_row[i])
        tp += m["tp"]
        fp += m["fp"]
        fn += m["fn"]

    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return p, r, f


# Only the rows every run actually covered.
compare_rows = sorted(set(zero_by_row) & set(theirs_by_row) & set(ours_by_row))
print("Comparing on", len(compare_rows), "flows")
print()

print("%-24s %10s %8s %8s" % ("", "Precision", "Recall", "F1"))
for name, by_row in [("Zero shot", zero_by_row),
                     ("One shot (theirs)", theirs_by_row),
                     ("One shot (controlled)", ours_by_row)]:
    p, r, fsc = score_rows(by_row, compare_rows)
    print("%-24s %10.3f %8.3f %8.3f" % (name, p, r, fsc))

Comparing on 19 flows

                          Precision   Recall       F1
Zero shot                     0.193    0.233    0.211
One shot (theirs)             0.629    0.767    0.691
One shot (controlled)         0.628    0.788    0.699


### Few shot prompting

In [ ]:
# Their three worked examples, by row number.
EXAMPLE_ROWS = [0, 3, 14]   # signal_system_configuration, reversible_lane_control, traffic_image_meta_data

print("Examples used:")
for row in EXAMPLE_ROWS:
    print(" ", row, df["flow_id"][row])


def build_few_shot_prompt(flow_description):
    """Their zero shot wording, plus three worked examples."""

    # Build the example block first.
    examples_text = ""
    for number, row in enumerate(EXAMPLE_ROWS, start=1):
        examples_text += f'''
                    -------------------------------------------------------------------
                    Example {number}:
                    {df["TEXT-flow_fn_threat"][row]}
                    -------------------------------------------------------------------
                    Classified Labels in python list format:
                    {df["str_label"][row]}
'''

    return f'''I am trying to do a multilabel classification of information flow description from Intelligent Transportation System (ITS) to MITRE ATT&CK Techniques.
                Here we have information flow name, its initiator and acceptor (Physcial Object), functional objects and process description associated with it, security characteristics of the information flow and the description of the information flow itself.
                We also have the STRIDE based threat information for the information flow.
                An attacker may attempt to compromise the integrity, confidentiality, or availability of the information flow in many ways.
                Find all the relevant MITRE ATT&CK techniques that the attacker might use to attack the information flow.

                Follow the instructions below carefully.

                1. We have a predefined list of MITRE ATT&CK Techniques that consists of 63 MITRE Techniques. You have to choose only the relevant MITRE ATT&CK Techniques from this list that is relevant to the information flow given.

                2. Understand the entire context, then generate a sublist of MITRE ATT&CK Technique from the given list.

                3. Do not add any other description in your answer.

                4. Only return the Technique IDs in python list format

                Given MITRE Technique List = {TECHNIQUE_LIST}
                You should return MITRE Techniques from this list.

                Here are a few examples:
{examples_text}
                    Now, here is your information flow description:
                    {flow_description}

                    Which are the relevant MITRE ATT&CK Techniques from the given list that the attacker might use to attack the information flow? Return the Technique IDs in python list format.
                    '''


# Never test on a flow we used as an example.
few_shot_test_indexes = [i for i in range(1, min(HOW_MANY + 1, len(df))) if i not in EXAMPLE_ROWS]

print()
print("Testing on", len(few_shot_test_indexes), "flows")
print("Prompt size:", count_tokens(build_few_shot_prompt("")), "tokens")

Examples used:
  0 2B_TM16-signal_system_configuration
  3 2B_TM16-reversible_lane_control
  14 2B_TM16-traffic_image_meta_data

Testing on 18 flows
Prompt size: 4624 tokens


In [ ]:
few_shot_predictions = []

for i in few_shot_test_indexes:
    prompt = build_few_shot_prompt(df["TEXT-flow_fn_threat"][i])

    answer = query_gpt(prompt)
    predicted = parse_mitre_techniques(answer)
    few_shot_predictions.append(predicted)

    print(i, df["flow_id"][i], "->", len(predicted), "techniques")

print()
print("Done.", len(few_shot_predictions), "flows classified.")

1 2A_TM16-traffic_detector_coordination -> 7 techniques
2 2A_TM16-traffic_detector_coordination -> 7 techniques
4 2B_TM16-video_surveillance_control -> 7 techniques
5 2B_TM16-signal_control_device_configuration -> 7 techniques
6 _TM16-vehicle_characteristics -> 6 techniques
7 2B_TM16-traffic_detector_control -> 7 techniques
8 2A_TM16-lane_management_coordination -> 7 techniques
9 2A_TM16-lane_management_coordination -> 7 techniques
10 2B_TM16-signal_control_plans -> 7 techniques
11 _TM16-traffic_operator_data -> 4 techniques
12 2A_TM16-signal_control_coordination -> 7 techniques
13 2A_TM16-signal_control_coordination -> 7 techniques
15 2B_TM16-reversible_lane_status -> 7 techniques
16 _PT04-traveler_interface_updates -> 4 techniques
17 2C_PT09-traffic_control_priority_status -> 7 techniques
18 2C_PT09-traffic_control_priority_request -> 7 techniques
19 2B_PT09-signal_control_commands -> 7 techniques
20 2A_PT09-local_signal_priority_request -> 7 techniques

Done. 18 flows classified.


In [ ]:
few_by_row = {i: few_shot_predictions[pos] for pos, i in enumerate(few_shot_test_indexes)}

compare_rows = sorted(set(zero_by_row) & set(theirs_by_row) & set(ours_by_row) & set(few_by_row))
print("Comparing on", len(compare_rows), "flows")
print()

print("%-24s %10s %8s %8s" % ("", "Precision", "Recall", "F1"))
for name, by_row in [("Zero shot", zero_by_row),
                     ("One shot (theirs)", theirs_by_row),
                     ("One shot (controlled)", ours_by_row),
                     ("Few shot (3 examples)", few_by_row)]:
    p, r, fsc = score_rows(by_row, compare_rows)
    print("%-24s %10.3f %8.3f %8.3f" % (name, p, r, fsc))

Comparing on 17 flows

                          Precision   Recall       F1
Zero shot                     0.196    0.235    0.214
One shot (theirs)             0.633    0.758    0.690
One shot (controlled)         0.632    0.780    0.698
Few shot (3 examples)         0.875    0.742    0.803


### Few shot, their exact wording

In [ ]:
# Find their few shot cell: it shows several examples.
few_shot_cell = None
for cell in their_nb["cells"]:
    if cell["cell_type"] != "code":
        continue
    source = "".join(cell["source"])
    if "Here are a few examples" in source:
        few_shot_cell = source
        break

# Pull the prompt text out from between the triple quotes.
start = few_shot_cell.index("final_prompt =")
start = few_shot_cell.index("'''", start) + 3
end = few_shot_cell.index("'''", start)
their_few_shot_template = few_shot_cell[start:end]


def build_their_few_shot_prompt(flow_description):
    """Their few shot prompt, verbatim from their own notebook."""
    return their_few_shot_template.replace("{prompt}", flow_description)


# Their three examples are these flows, so we never test on them.
EXAMPLE_ROWS = [0, 3, 14]
few_shot_test_indexes = [i for i in range(1, HOW_MANY + 1) if i not in EXAMPLE_ROWS]

print("Characters:", len(their_few_shot_template))
print("Tokens:", count_tokens(their_few_shot_template))
print()
print("Examples baked into their prompt:")
for row in EXAMPLE_ROWS:
    print(" ", row, df["flow_id"][row])
print()
print("Testing on", len(few_shot_test_indexes), "flows")

Characters: 34623
Tokens: 6251

Examples baked into their prompt:
  0 2B_TM16-signal_system_configuration
  3 2B_TM16-reversible_lane_control
  14 2B_TM16-traffic_image_meta_data

Testing on 18 flows


In [ ]:
few_shot_predictions = []

for i in few_shot_test_indexes:
    prompt = build_their_few_shot_prompt(df["TEXT-flow_fn_threat"][i])

    answer = query_gpt(prompt)
    predicted = parse_mitre_techniques(answer)
    few_shot_predictions.append(predicted)

    print(i, df["flow_id"][i], "->", len(predicted), "techniques")

print()
print("Done.", len(few_shot_predictions), "flows classified.")

1 2A_TM16-traffic_detector_coordination -> 7 techniques
2 2A_TM16-traffic_detector_coordination -> 7 techniques
4 2B_TM16-video_surveillance_control -> 7 techniques
5 2B_TM16-signal_control_device_configuration -> 7 techniques
6 _TM16-vehicle_characteristics -> 7 techniques
7 2B_TM16-traffic_detector_control -> 7 techniques
8 2A_TM16-lane_management_coordination -> 7 techniques
9 2A_TM16-lane_management_coordination -> 7 techniques
10 2B_TM16-signal_control_plans -> 7 techniques
11 _TM16-traffic_operator_data -> 6 techniques
12 2A_TM16-signal_control_coordination -> 7 techniques
13 2A_TM16-signal_control_coordination -> 7 techniques
15 2B_TM16-reversible_lane_status -> 7 techniques
16 _PT04-traveler_interface_updates -> 4 techniques
17 2C_PT09-traffic_control_priority_status -> 7 techniques
18 2C_PT09-traffic_control_priority_request -> 7 techniques
19 2B_PT09-signal_control_commands -> 7 techniques
20 2A_PT09-local_signal_priority_request -> 7 techniques

Done. 18 flows classified.


In [ ]:
few_by_row = {i: few_shot_predictions[pos] for pos, i in enumerate(few_shot_test_indexes)}

compare_rows = sorted(set(zero_by_row) & set(theirs_by_row) & set(ours_by_row) & set(few_by_row))
print("Comparing on", len(compare_rows), "flows")
print()

print("%-24s %10s %8s %8s" % ("", "Precision", "Recall", "F1"))
for name, by_row in [("Zero shot", zero_by_row),
                     ("One shot (theirs)", theirs_by_row),
                     ("One shot (controlled)", ours_by_row),
                     ("Few shot (theirs)", few_by_row)]:
    p, r, fsc = score_rows(by_row, compare_rows)
    print("%-24s %10.3f %8.3f %8.3f" % (name, p, r, fsc))

Comparing on 17 flows

                          Precision   Recall       F1
Zero shot                     0.196    0.235    0.214
One shot (theirs)             0.633    0.758    0.690
One shot (controlled)         0.632    0.780    0.698
Few shot (theirs)             0.875    0.742    0.803
